# E05 — Grid-serving flexibility

| | |
|---|---|
| **Question** | Can Vessim express what the edge study calls grid-serving flexibility: grid limits, demand-response events, a flexibility band and islanding? |
| **Hypothesis** | H3 (flexibility management is weak: greedy single-step dispatch, no look-ahead, no flexibility band, no forecast interface); side results for H4 |
| **Branch** | `exp/e05-grid-flexibility` |
| **Date** | 2026-11-13 |
| **Site** | Edge study S1.2 (2025-04-01, one day, 15-min IT, cooling, building and PV from the E01 run), edge battery (500 kWh, ±250 kW, 15–90 %, 95 %/95 %) |

**Parts**

| Part | What is tested | How it is built in Vessim |
|---|---|---|
| A. Grid limits | import cap and ramp limits on `GridConnection` | built-in `DefaultDispatchPolicy`, two dispatch orders; no custom code |
| B. Demand-response event | import cap 4 200 kW, 17:00–19:00, announced one step ahead | as in Vessim's `examples/demand_response.py`: a `Controller` sets `GridConnection.max_import` from a grid signal; custom `GridServingPolicy` uses the battery and a `DeferrableLoad` |
| C. Flexibility band | minimum / reference / maximum grid import per step | custom `FlexibilityBand` controller built on the resources' own `feasible_range(duration)`; compared with the edge study's `flexibility_results.csv` and checked by activating the band |
| D. Islanding | grid disconnected 11:30–12:30 | `GridConnection.connected = False` from the same controller; optional reconnection ramp |

The flexible load is the delay-tolerant share of the IT and cooling power (20 %, the delay-tolerant energy share of the
edge job trace in E03), modelled as a Vessim `Dispatchable` so that the dispatch policy can see and move it.

## 1. Setup

In [1]:
import importlib.metadata as md_
import time

import numpy as np
import pandas as pd
from IPython.display import display
from loguru import logger

from vessim_eval import RESULTS_DIR
from vessim_eval import plotting as vp
from vessim_eval.dispatch import logic_lines, optimal_schedule, price_from_reference
from vessim_eval.edge_reference import load_reference, scenario_config
from vessim_eval.flexibility import (BalanceSlack, DeferrableLoad, FlexibilityBand, FlexParams,
                                     FlexSite, GridEventController, GridServingPolicy, GridSpec,
                                     activation_check, first_error, grid_kpis, lowest_feasible,
                                     run_grid_site, window_series)
from vessim_eval.scenarios import BatteryParams

logger.remove()  # Vessim logs every run start; keep the notebook output readable
print("vessim", md_.version("vessim"))

vessim 0.16.0


In [2]:
# ---- Parameters -------------------------------------------------------------
STEP_S = 900                                   # edge physical step
FLEX_SHARE = 0.20                              # delay-tolerant energy share of the edge job trace (E03: 19.8 %)
FLEX = FlexParams(max_backlog_kwh=1000.0,      # at most 1 MWh of work owed (≈ 1 h of the flexible share)
                  max_catchup_kw=500.0)        # spare capacity to catch up deferred work
CAP_SWEEP_KW = [6300, 6200, 6100, 6050, 6000, 5950]
RAMP_STEPS_S = {"15 min": 900, "5 min": 300, "1 min": 60}
PHYSICAL_RAMP_KW_PER_15MIN = 200.0             # one physical ramp rate tried at every step size
EVENT = ("17:00", "19:00")                     # evening demand-response event
EVENT_CAP_KW = 4200.0
RECOVERY_CAP_KW = 5000.0                       # optional cap on catch-up / recharge after the event
ISLAND = ("11:30", "12:30")
RECONNECT_RAMP_KW = 1000.0                     # per 15-min step
BAND_DURATIONS_S = [120, 900, 3600, 4 * 3600]  # 120 s = edge band horizon (4 scheduler steps of 30 s)
ACTIVATION_STARTS = ["06:00", "12:00", "20:00"]
ACTIVATION_S = 3600
OUT = RESULTS_DIR / "E05"
OUT.mkdir(parents=True, exist_ok=True)
# No randomness is involved (no seed needed).

ref = load_reference("e01_edge_s1_2_15min.csv")
price = price_from_reference(ref)
dc_cfg = scenario_config("s1_2_battery_pv_large")["data_centers"][0]
bat = BatteryParams.from_edge_config(dc_cfg["battery_config"])
site = FlexSite.from_reference(ref, price, FLEX_SHARE)
net = site.net_load_kw
edge_band = load_reference("e05_edge_s1_2_flexibility_15min.csv")

pd.Series({
    "Net site demand, mean (kW)": net.mean(),
    "Net site demand, peak (kW)": net.max(),
    "Largest 15-min increase of net demand (kW)": net.diff().max(),
    "Largest 15-min decrease of net demand (kW)": -net.diff().min(),
    "Flexible share of demand, mean (kW)": site.flexible_kw.mean(),
    "PV peak (kW)": site.pv_kw.max(),
    "Edge grid connection max_power (kW)": dc_cfg["grid_config"]["max_power"],
}, name="S1.2, 2025-04-01").to_frame().style.format("{:,.1f}")

,"S1.2, 2025-04-01"
"Net site demand, mean (kW)","4,589.9"
"Net site demand, peak (kW)","6,255.3"
Largest 15-min increase of net demand (kW),862.2
Largest 15-min decrease of net demand (kW),308.8
"Flexible share of demand, mean (kW)",941.8
PV peak (kW),593.7
Edge grid connection max_power (kW),"10,000.0"


**Flexible load model.** `DeferrableLoad` runs at a reference power (20 % of IT + cooling). It may run below the
reference while the backlog of deferred work stays below 1 MWh, and above it by up to 500 kW while there is backlog to
catch up. It cannot run ahead of its reference. The backlog has no deadline.

This is an **upper bound** on job flexibility: it treats delay-tolerant work as divisible and interruptible at no cost.
The edge study moves whole, non-preemptive jobs, and in E03 shifting the same jobs lowered the 15-min peak by at most
385 kW. The flexible load is a `Dispatchable`, not an actor, because the dispatch policy only sees dispatchables.
Dispatchables get no timestamp, so the load keeps its own clock from the durations passed to `step()`.

**Unmet demand.** Vessim requires an exact balance and raises `RuntimeError` otherwise. To measure how much demand a
constraint costs instead of stopping at the first violation, the custom runs add a `BalanceSlack` dispatchable that
records unserved (and spilled) power.

## 2. Run

In [3]:
t_start, walls = time.perf_counter(), []

def run(**kw):
    r = run_grid_site(site, bat, **kw)
    walls.append(r.wall_s)
    return r

GF, BF = ("grid", "battery"), ("battery", "grid")
custom = lambda **kw: GridServingPolicy(soc_target=bat.soc_max, **kw)  # noqa: E731

# A1. Import cap with the built-in policy
caps = {(o, c): run(order=order, grid=GridSpec(max_import_kw=c))
        for o, order in (("battery first", BF), ("grid first", GF)) for c in CAP_SWEEP_KW}
lowest_cap = {o: lowest_feasible(lambda c, order=order: run(order=order, grid=GridSpec(max_import_kw=c)),
                                 5000, 6300, tol=1)
              for o, order in (("battery first", BF), ("grid first", GF))}
lp_peak = (net - optimal_schedule(net, price, bat, objective="peak")).max()

# A2. Ramp limits (Vessim: W per dispatch step, both directions)
n0 = net.iloc[0]
ramp_no_initial = run(order=GF, grid=GridSpec(ramp_kw_per_step=1000))     # initial_power left at 0
ramp_lowest = {}
for o, order in (("battery first", BF), ("grid first", GF)):
    for lbl, st in RAMP_STEPS_S.items():
        per_step = lowest_feasible(lambda r_, order=order, st=st: run(
            order=order, grid=GridSpec(ramp_kw_per_step=r_, initial_import_kw=n0), step_s=st), 0, 2000, tol=2)
        same_rate = lowest_feasible(lambda r_, order=order, st=st: run(
            order=order, grid=GridSpec(ramp_kw_per_step=r_ * st / 900, initial_import_kw=n0), step_s=st),
            0, 2000 * 900 / st, tol=2)
        ramp_lowest[(o, lbl)] = (per_step, same_rate)
ramp_custom = {(lbl, scaling): run(policy=custom(), flexible=FLEX, slack=True, step_s=st,
                                   grid=GridSpec(ramp_kw_per_step=PHYSICAL_RAMP_KW_PER_15MIN * (st / 900 if scaling else 1),
                                                 initial_import_kw=n0))
               for lbl, st in RAMP_STEPS_S.items() for scaling in (False, True)}

# B. Demand-response event (cap announced one step ahead)
lim = window_series(net.index, [(*EVENT, EVENT_CAP_KW)], np.nan)
dr = {
    "Built-in policy, battery first": run(import_limit_kw=lim),
    "Built-in policy, grid first": run(order=GF, import_limit_kw=lim),
    "Custom policy, battery only, no slack": run(policy=custom(use=("battery",)), flexible=FLEX, import_limit_kw=lim),
    "No event": run(policy=custom(), flexible=FLEX, slack=True),
    "Battery only": run(policy=custom(use=("battery",)), flexible=FLEX, slack=True, import_limit_kw=lim),
    "Flexible load only": run(policy=custom(use=("flexible",)), flexible=FLEX, slack=True, import_limit_kw=lim),
    "Battery + flexible load": run(policy=custom(), flexible=FLEX, slack=True, import_limit_kw=lim,
                                   band_durations_s=BAND_DURATIONS_S),
    "Battery + flexible load, load used first": run(policy=custom(shortfall_order=("flexible", "battery")),
                                                    flexible=FLEX, slack=True, import_limit_kw=lim),
    "Battery + flexible load, not announced": run(policy=custom(), flexible=FLEX, slack=True, import_limit_kw=lim,
                                                  announce=False),
    "Battery + flexible load, capped recovery": run(policy=custom(recovery_cap_kw=RECOVERY_CAP_KW), flexible=FLEX,
                                                    slack=True, import_limit_kw=lim),
}

# C. Flexibility band
band_runs = {
    "Edge operating point: battery idle at 50 %, no flexible load": run(policy=GridServingPolicy(use=()),
                                                                         band_durations_s=BAND_DURATIONS_S),
    "Battery held at 90 % + flexible load": run(policy=custom(), flexible=FLEX, slack=True,
                                                band_durations_s=BAND_DURATIONS_S),
    "Battery held at 50 % + flexible load": run(policy=GridServingPolicy(soc_target=0.5), flexible=FLEX, slack=True,
                                                band_durations_s=BAND_DURATIONS_S),
    "Demand-response event, battery + flexible load": dr["Battery + flexible load"],
}
activation = activation_check(
    lambda act: run(policy=GridServingPolicy(soc_target=0.5), flexible=FLEX, slack=True, activation=act),
    band_runs["Battery held at 50 % + flexible load"], ACTIVATION_STARTS, ACTIVATION_S, compare_s=900)

# D. Islanding (disconnection announced one step ahead)
con = window_series(net.index, [(*ISLAND, 0)], 1).astype(bool)
island = {
    "Built-in policy, battery first": run(connected=con),
    "Built-in policy, grid first": run(order=GF, connected=con),
    "Island, no reconnection ramp": run(policy=custom(), flexible=FLEX, slack=True, connected=con),
    "Island, reconnection ramp 1 000 kW per step": run(policy=custom(), flexible=FLEX, slack=True, connected=con,
                                                       grid=GridSpec(ramp_kw_per_step=RECONNECT_RAMP_KW,
                                                                     initial_import_kw=n0)),
}
wall = time.perf_counter() - t_start
print(f"{len(walls)} Vessim runs: {sum(walls):.2f} s simulation, {wall:.1f} s cell wall time; "
      f"median {1e3 * np.median(walls):.1f} ms per run, max {1e3 * max(walls):.0f} ms")

210 Vessim runs: 3.86 s simulation, 5.5 s cell wall time; median 6.6 ms per run, max 182 ms


In [4]:
# Sanity checks: limits hold wherever the run completed and the limit was announced in time
for name, r in dr.items():
    if r.error is None and "not announced" not in name and name != "No event":
        assert (r.ts.grid_kw <= r.ts.import_limit_kw.fillna(np.inf) + 1e-6).all(), name
for name, r in island.items():
    if r.error is None:
        assert (r.ts.grid_kw[~r.ts.connected].abs() < 1e-9).all(), name
assert (activation.iloc[:, 0] - activation.iloc[:, 1]).abs().max() < 10  # band edge delivered within 10 kW
print("checks passed")

checks passed


## 3. Results

### 3.1 Import limit with the built-in policy

`DefaultDispatchPolicy` allocates in list order. With the battery first (Vessim's usual order) the battery covers every
deficit until it is empty; with the grid first, the grid takes everything up to its limit and the battery covers only the
excess. The minimum-peak LP from E02 (perfect foresight) gives the lowest cap any schedule could meet.

In [5]:
cap_tab = pd.DataFrame({o: {f"{c:,.0f} kW": ("completed" if caps[(o, c)].error is None
                                             else f"fails at {caps[(o, c)].failed_at:%H:%M}")
                            for c in CAP_SWEEP_KW} for o in ("battery first", "grid first")})
cap_tab.index.name = "Import cap"
display(cap_tab)
lowest = pd.Series({
    "Peak without limit": net.max(),
    "Lowest feasible cap, built-in policy, battery first": lowest_cap["battery first"],
    "Lowest feasible cap, built-in policy, grid first": lowest_cap["grid first"],
    "Minimum peak, perfect-foresight LP (E02)": lp_peak,
}, name="Grid import (kW)").to_frame()
lowest.to_csv(OUT / "import_cap.csv")
display(lowest.style.format("{:,.1f}"))
print("Error at 6 000 kW, battery first:", caps[("battery first", 6000)].error)

,battery first,grid first
Import cap,,
"6,300 kW",completed,completed
"6,200 kW",fails at 03:45,completed
"6,100 kW",fails at 03:45,completed
"6,050 kW",fails at 03:45,completed
"6,000 kW",fails at 03:45,fails at 04:00
"5,950 kW",fails at 03:45,fails at 03:45


,Grid import (kW)
Peak without limit,"6,255.3"
"Lowest feasible cap, built-in policy, battery first","6,255.6"
"Lowest feasible cap, built-in policy, grid first","6,039.7"
"Minimum peak, perfect-foresight LP (E02)","6,005.3"


Error at 6 000 kW, battery first: RuntimeError: Unmet demand of 238568 W: dispatch must balance exactly.


In [6]:
cap_kw = 6050
r_bf, r_gf = caps[("battery first", cap_kw)], caps[("grid first", cap_kw)]
fig = vp.stacked_panels([
    ("Grid import (kW)", {"Net site demand": net, "Built-in policy, battery first": r_bf.ts.grid_kw,
                          "Built-in policy, grid first": r_gf.ts.grid_kw,
                          "Import limit": pd.Series(cap_kw, index=net.index)}),
    ("Battery state of charge (%)", {"Built-in policy, battery first": 100 * r_bf.ts.soc,
                                     "Built-in policy, grid first": 100 * r_gf.ts.soc}),
], title=f"Built-in dispatch under a {cap_kw:,} kW import cap (battery first stops at {r_bf.failed_at:%H:%M})",
   row_height=260)
vp.save(fig, OUT / "import_limit", png=True)
fig.show()

### 3.2 Ramp limits

Vessim's `ramp_up` / `ramp_down` are in W **per dispatch step**, not per second, and also apply to the first step from
`initial_power` (default 0). The table gives the lowest symmetric ramp limit for which the built-in policy completes the
day, once with the same value per step at every step size and once with the same physical rate (kW per 15 min).

In [7]:
print("initial_power left at 0:", ramp_no_initial.error, f"(at {ramp_no_initial.failed_at:%H:%M})")
ramp_tab = pd.DataFrame({
    (o, lbl): {"Lowest feasible ramp, same value per step (kW per step)": v[0],
               "Lowest feasible ramp, same physical rate (kW per 15 min)": v[1],
               "Same physical rate, as kW per step": v[1] * RAMP_STEPS_S[lbl] / 900}
    for (o, lbl), v in ramp_lowest.items()}).T
ramp_tab.index.names = ["Built-in policy", "Step"]
ramp_tab.to_csv(OUT / "ramp_limits.csv")
display(vp.format_table(ramp_tab, 0))

rc = pd.DataFrame({
    (lbl, "same rate" if scaling else "same value per step"): grid_kpis(r)[
        ["Unserved energy (kWh)", "Flexible load deferred (kWh)", "Battery discharge (kWh)"]]
    for (lbl, scaling), r in ramp_custom.items()}).T
rc["Spilled energy (kWh)"] = [r.ts.spilled_kw.sum() * r.attrs["step_s"] / 3600 for r in ramp_custom.values()]
rc.index.names = ["Step", f"Ramp {PHYSICAL_RAMP_KW_PER_15MIN:.0f} kW per 15 min applied as"]
display(vp.format_table(rc.astype(float), 1))

initial_power left at 0: RuntimeError: Unmet demand of 1.09248e+06 W: dispatch must balance exactly. (at 00:00)


### 3.3 Demand-response event: import cap 4 200 kW, 17:00–19:00

The cap is published as grid signal `next_import_limit_w` one step ahead; `GridEventController` writes it into
`GridConnection.max_import` after the dispatch of the step before the event (Vessim's `demand_response.py` pattern).
In the "not announced" variant the signal carries the cap of the current interval, so it takes effect one step late.

In [8]:
over = (net - EVENT_CAP_KW).where(lim.notna()).clip(lower=0)
print(f"Without flexibility the site would exceed the cap by up to {over.max():,.0f} kW "
      f"and {over.sum() * STEP_S / 3600:,.0f} kWh during the event.")
for n in ("Built-in policy, battery first", "Built-in policy, grid first", "Custom policy, battery only, no slack"):
    print(f"{n}: {dr[n].error} (at {dr[n].failed_at:%H:%M})")
dr_tab = pd.DataFrame({n: grid_kpis(r, EVENT) for n, r in dr.items() if r.error is None}).T
base_cost = dr_tab.loc["No event", "Energy cost (€)"]
dr_tab.insert(3, "Cost change vs no event (€)", dr_tab["Energy cost (€)"] - base_cost)
dr_tab = dr_tab.drop(columns=["Completed", "Failed at", "Peak grid import (kW)"]).astype(float)
dr_tab.index.name = "Variant"
dr_tab.to_csv(OUT / "demand_response.csv")
vp.format_table(dr_tab, 1)

Without flexibility the site would exceed the cap by up to 702 kW and 1,224 kWh during the event.
Built-in policy, battery first: RuntimeError: Unmet demand of 474045 W: dispatch must balance exactly. (at 17:00)
Built-in policy, grid first: RuntimeError: Unmet demand of 224045 W: dispatch must balance exactly. (at 17:00)
Custom policy, battery only, no slack: RuntimeError: Unmet demand of 224045 W: grid limit reached. (at 17:00)


,Cost change vs no event (€),Grid import energy (kWh),Energy cost (€),Import above limit (kWh),Unserved energy (kWh),Battery discharge (kWh),Flexible load deferred (kWh),Backlog left at end (kWh),Final state of charge (%),Max import in window (kW),Max import in 2 h after window (kW)
Variant,,,,,,,,,,,
No event,0.0,"110,368.0","10,482.2",0.0,0.0,0.0,0.0,0.0,90.0,"4,901.7","4,914.9"
Battery only,-128.2,"109,538.9","10,354.0",0.0,867.6,356.2,0.0,0.0,90.0,"4,200.0","5,138.0"
Flexible load only,-90.2,"109,933.7","10,392.1",0.0,223.8,0.0,"1,000.0",0.0,50.0,"4,200.0","5,414.9"
Battery + flexible load,-41.9,"110,406.5","10,440.3",0.0,0.0,356.2,867.6,0.0,90.0,"4,200.0","5,638.0"
"Battery + flexible load, load used first",-56.4,"110,274.3","10,425.8",0.0,106.4,117.4,"1,000.0",0.0,90.0,"4,200.0","5,578.8"
"Battery + flexible load, not announced",-42.7,"110,406.5","10,439.5",118.5,0.0,356.2,904.6,0.0,90.0,"4,674.0","5,648.3"
"Battery + flexible load, capped recovery",-94.0,"110,074.3","10,388.2",0.0,0.0,356.2,867.6,0.0,26.9,"4,200.0","5,000.0"


In [9]:
win = slice("2025-04-01 14:00", "2025-04-01 23:45")
show = ["No event", "Battery only", "Flexible load only", "Battery + flexible load",
        "Battery + flexible load, not announced", "Battery + flexible load, capped recovery"]
fig = vp.stacked_panels([
    ("Grid import (kW)", {**{n: dr[n].ts.grid_kw[win] for n in show}, "Import limit": lim[win]}),
    ("Unserved demand (kW)", {n: dr[n].ts.unserved_kw[win] for n in show[1:4]}),
    ("Battery state of charge (%)", {n: 100 * dr[n].ts.soc[win] for n in show}),
    ("Deferred work owed (kWh)", {n: dr[n].ts.backlog_kwh[win] for n in show}),
], title="Demand response: 4 200 kW import cap, 17:00–19:00", row_height=230)
vp.save(fig, OUT / "demand_response", png=True)
fig.show()

### 3.4 Flexibility band

`FlexibilityBand` records, after each dispatch, how far grid import could move for a sustained duration *D*: each
resource's `feasible_range(D)` minus its current setpoint, summed and clipped to the grid's import limit and connection
state. *D* = 2 min is the edge study's band horizon (4 scheduler steps of 30 s). The edge band is taken from
`flexibility_results.csv` of the E01 S1.2 run (horizon step 0, averaged per 15 min, `data/e05_edge_s1_2_flexibility_15min.csv`).

In [10]:
def band_stats(b, d):
    return {"Downward, mean (kW)": (b.p_ref_w - b[f"p_min_w_{d}"]).mean(),
            "Downward, min (kW)": (b.p_ref_w - b[f"p_min_w_{d}"]).min(),
            "Upward, mean (kW)": (b[f"p_max_w_{d}"] - b.p_ref_w).mean(),
            "Upward, min (kW)": (b[f"p_max_w_{d}"] - b.p_ref_w).min()}

DUR_LABEL = {120: "2 min", 900: "15 min", 3600: "1 h", 14400: "4 h"}
rows = {}
for src, d_col, u_col in (("job band", "job_down_kw", "job_up_kw"), ("battery band", "bat_down_kw", "bat_up_kw"),
                          ("offline-optimal battery band", "batopt_down_kw", "batopt_up_kw")):
    rows[("Edge study", src, "2 min")] = {"Downward, mean (kW)": edge_band[d_col].mean(),
                                          "Downward, min (kW)": edge_band[d_col].min(),
                                          "Upward, mean (kW)": edge_band[u_col].mean(),
                                          "Upward, min (kW)": edge_band[u_col].min()}
for name, r in band_runs.items():
    for d in BAND_DURATIONS_S:
        rows[("Vessim", name, DUR_LABEL[d])] = band_stats(r.band, d)
band_tab = pd.DataFrame(rows).T
band_tab.index.names = ["Tool", "Case", "Sustained for"]
band_tab.to_csv(OUT / "flexibility_band.csv")
display(vp.format_table(band_tab, 0))
off = edge_band.p_ref_h1_kw - edge_band.p_ref_kw
print(f"Edge band reference: horizon step 1 (30 s ahead) minus step 0 (observed): mean {off.mean():,.0f} kW, "
      f"range {off.min():,.0f} to {off.max():,.0f} kW")

Edge band reference: horizon step 1 (30 s ahead) minus step 0 (observed): mean -464 kW, range -1,171 to 207 kW


In [11]:
def vband(r, durations):
    b = r.band
    return {f"Vessim, sustained {DUR_LABEL[d]}": (b[f"p_min_w_{d}"], b[f"p_max_w_{d}"]) for d in durations}

edge_ref = edge_band.p_ref_kw
fig = vp.band_panels([
    ("Edge study band (battery + jobs, 2-min horizon)",
     edge_ref, {"Edge study, battery + jobs": (edge_ref - edge_band.bat_down_kw - edge_band.job_down_kw,
                                               edge_ref + edge_band.bat_up_kw + edge_band.job_up_kw)}),
    ("Vessim, edge operating point (battery idle at 50 %, no flexible load)",
     band_runs["Edge operating point: battery idle at 50 %, no flexible load"].band.p_ref_w,
     vband(band_runs["Edge operating point: battery idle at 50 %, no flexible load"], [120, 3600, 14400])),
    ("Vessim, battery held at 50 % + flexible load",
     band_runs["Battery held at 50 % + flexible load"].band.p_ref_w,
     vband(band_runs["Battery held at 50 % + flexible load"], [120, 3600, 14400])),
    ("Vessim, demand-response event 17:00–19:00 (battery held at 90 % + flexible load)",
     band_runs["Demand-response event, battery + flexible load"].band.p_ref_w,
     vband(band_runs["Demand-response event, battery + flexible load"], [120, 3600, 14400])),
], title="Flexibility band of grid import: edge study vs Vessim for several sustain durations",
   y_title="Grid import (kW)", row_height=250)
vp.save(fig, OUT / "flexibility_band", png=True)
fig.show()

**Can the band be delivered?** Starting from the run with the battery held at 50 %, all flexibility is activated for
1 h (signal `flex_activation` = −1 or +1) at three times of day. "Delivered" is the mean change of grid import over the
hour compared with the run without activation.

In [12]:
activation.to_csv(OUT / "band_activation.csv")
vp.format_table(activation.astype(float), 1)

### 3.5 Islanding: grid disconnected 11:30–12:30

In [13]:
for n in ("Built-in policy, battery first", "Built-in policy, grid first"):
    print(f"{n}: {island[n].error} (at {island[n].failed_at:%H:%M})")

def island_kpis(r):
    ts, dt = r.ts, r.attrs["step_s"] / 3600
    tod = ts.index - ts.index.normalize()
    isl = (tod >= pd.Timedelta(ISLAND[0] + ":00")) & (tod < pd.Timedelta(ISLAND[1] + ":00"))
    after = (tod >= pd.Timedelta(ISLAND[1] + ":00"))
    demand = (ts.inflexible_kw + ts.flexible_ref_kw)[isl].sum() * dt
    return pd.Series({
        "Site demand during island (kWh)": demand,
        "PV during island (kWh)": ts.pv_kw[isl].sum() * dt,
        "Battery discharge during island (kWh)": ts.battery_kw[isl].clip(lower=0).sum() * dt,
        "Flexible load deferred during island (kWh)": (ts.flexible_ref_kw - ts.flexible_kw)[isl].sum() * dt,
        "Unserved during island (kWh)": ts.unserved_kw[isl].sum() * dt,
        "Share of demand served during island (%)": 100 * (1 - ts.unserved_kw[isl].sum() * dt / demand),
        "Unserved after reconnection (kWh)": ts.unserved_kw[after].sum() * dt,
        "Time to full supply after reconnection (min)": (
            (ts.index[after & (ts.unserved_kw > 1e-6)].max() - ts.index[after][0]).total_seconds() / 60 + 15
            if (ts.unserved_kw[after] > 1e-6).any() else 0.0),
        "Max import in 2 h after reconnection (kW)": ts.grid_kw[after][:8].max(),
    })

isl_tab = pd.DataFrame({n: island_kpis(r) for n, r in island.items() if r.error is None}).T
isl_tab.index.name = "Variant"
isl_tab.to_csv(OUT / "islanding.csv")
vp.format_table(isl_tab, 1)

Built-in policy, battery first: RuntimeError: Unmet demand of 4.00526e+06 W: dispatch must balance exactly. (at 11:30)
Built-in policy, grid first: RuntimeError: Unmet demand of 3.75526e+06 W: dispatch must balance exactly. (at 11:30)


,Site demand during island (kWh),PV during island (kWh),Battery discharge during island (kWh),Flexible load deferred during island (kWh),Unserved during island (kWh),Share of demand served during island (%),Unserved after reconnection (kWh),Time to full supply after reconnection (min),Max import in 2 h after reconnection (kW)
Variant,,,,,,,,,
"Island, no reconnection ramp","4,506.2",590.0,250.0,898.3,"2,767.9",38.6,0.0,0.0,"4,661.5"
"Island, reconnection ramp 1 000 kW per step","4,506.2",590.0,250.0,898.3,"2,767.9",38.6,"1,201.1",45.0,"4,856.9"


In [14]:
win = slice("2025-04-01 10:00", "2025-04-01 16:00")
show = ["Island, no reconnection ramp", "Island, reconnection ramp 1 000 kW per step"]
fig = vp.stacked_panels([
    ("Grid import (kW)", {"Net site demand": net[win], **{n: island[n].ts.grid_kw[win] for n in show}}),
    ("Unserved demand (kW)", {n: island[n].ts.unserved_kw[win] for n in show}),
    ("Battery state of charge (%)", {n: 100 * island[n].ts.soc[win] for n in show}),
], title="Islanding 11:30–12:30 with load shedding, with and without a reconnection ramp", row_height=240)
vp.save(fig, OUT / "islanding", png=True)
fig.show()

### 3.6 Custom code needed

In [15]:
pd.Series({
    "Deferrable load as a dispatchable (DeferrableLoad)": logic_lines(DeferrableLoad),
    "Slack for unserved / spilled power (BalanceSlack)": logic_lines(BalanceSlack),
    "Limit-aware policy with recovery and activation (GridServingPolicy)": logic_lines(GridServingPolicy),
    "Import limit / connection from grid signals (GridEventController)": logic_lines(GridEventController),
    "Flexibility band recorder (FlexibilityBand)": logic_lines(FlexibilityBand),
}, name="Lines of custom code (no comments or docstrings)").to_frame()

,Lines of custom code (no comments or docstrings)
Deferrable load as a dispatchable (DeferrableLoad),28
Slack for unserved / spilled power (BalanceSlack),9
Limit-aware policy with recovery and activation (GridServingPolicy),77
Import limit / connection from grid signals (GridEventController),10
Flexibility band recorder (FlexibilityBand),22


### 3.7 Grid-serving capabilities: edge study vs Vessim

| Capability | Edge study (`3e8533a`) | Vessim 0.16.0 |
|---|---|---|
| Import limit | `grid_config.max_power` (10 MW in S1.2, never binding); `simulators/power_grid_connection.py` raises `ValueError` if exceeded; the scheduler keeps planned IT load below the headroom (`scheduler.py`, constraint C4) | `GridConnection.max_import`; dispatch raises `RuntimeError` if it cannot be met (3.1) |
| Export limit | none (feed-in tariff only) | `GridConnection.max_export` |
| Ramp limits | none | `ramp_up` / `ramp_down`, per dispatch step (3.2) |
| Time-varying limit / demand response | none | change `max_import` from a `Controller`; takes effect at the next dispatch (3.3) |
| Islanding | none | `connected = False`, or a microgrid without `GridConnection` (3.5) |
| Unmet demand | error | error; shedding only through a user-defined slack resource |
| Flexibility band | yes: min / ref / max grid power for 4 × 30 s ahead, job band (MILP or greedy) + battery band + offline-optimal battery band, logged every scheduler call | none; a 22-line controller derives it from each resource's `feasible_range(duration)` for any duration (3.4) |
| Forecasts used for the band | price, PV, wind, building forecasts in the controller | none (no forecast interface) |
| Flexible load visible to dispatch | jobs in the scheduler | only as a user `Dispatchable`; actors are opaque to the policy |

## 4. Evaluation

- **H3 partially confirmed.** Vessim has first-class *hard* grid constraints that the edge study lacks: import and
  export limits, per-step ramps, and a connection flag, all changeable during a run. What H3 predicts is missing is
  indeed missing: no flexibility band, no forecast interface, no look-ahead, and no handling of unmet demand other than
  an exception. A band can be built cheaply on `feasible_range(duration)` (22 lines), and that band was delivered
  exactly when activated.
- **The import cap is only as good as the dispatch order.** With the battery first (Vessim's usual order) the battery
  is empty by 00:45, and every cap below the 6 255 kW peak aborts the run at 03:45. With the grid first, the same
  built-in policy becomes a peak-shaving rule without custom code: the lowest feasible cap is 6 040 kW, 35 kW above the
  perfect-foresight optimum (6 005 kW). It never recharges the battery, and too low a cap aborts the run.
- **Ramp limits are per dispatch step and start from `initial_power` = 0.** A ramp-limited connection fails at the
  first step unless `initial_power` is set. With 15-min input traces the binding case is the 862 kW load jump at a
  step boundary, so the lowest feasible ramp *per step* is the same at 15, 5 and 1 min:
  - battery first: 863 kW, the jump itself (battery already empty);
  - grid first: 613–631 kW, the jump minus the 250 kW battery.

  The same physical rate per 15 min is 3× (5 min) or 15× (1 min) tighter per step. With the custom policy, a 200 kW per
  15 min ramp leaves 184 kWh unserved at 15 min and 986 kWh at 1 min. Ramp values must be rescaled with the step size,
  and their effect depends on how the input traces are interpolated.
- **Unmet demand stops the simulation.** Every infeasible step raises `RuntimeError`. To measure what a constraint
  costs, a 9-line slack dispatchable is needed. The edge study behaves the same way (`ValueError` in
  `power_grid_connection.py`).
- **Demand response works with Vessim's own pattern, but the timing is the user's job.** A controller can only change
  `max_import` after a dispatch, so the cap must be announced one step ahead. Without notice, the first event interval
  exceeds the cap by 474 kW (118 kWh), and the cap stays one interval after the event ends.
- **Event outcome (cap 4 200 kW, 17:00–19:00, up to 702 kW and 1 224 kWh above the cap):**
  - battery only: 868 kWh unserved (250 kW power limit);
  - flexible load only: 224 kWh unserved (1 MWh backlog limit);
  - both: none.

  The order matters. Using the flexible load first fills its backlog before the event ends while the battery still
  holds energy, which leaves 106 kWh unserved. Splitting energy-limited resources over an event needs look-ahead;
  greedy single-step allocation cannot do it.
- **Rebound is unmanaged.** After the event, catch-up and recharge push import to 5 638 kW, 723 kW above the no-event
  day. Capping recovery at 5 MW removes the rebound, but the battery stays at 15 % until late evening and is not ready
  for a second event. Vessim offers nothing to balance this.
- **Flexibility band:**
  - At the edge operating point (battery idle at 50 %), Vessim's 2-min band reproduces the edge battery band
    (±250 kW).
  - Sustained for 1 h, the same battery offers only −166 / +211 kW; for 4 h, −42 / +53 kW. The edge band's 2-min
    horizon makes the battery look power-limited all day and says nothing about how long the flexibility lasts.
  - A 15-min band overstates the 1-h deliverable by 40–150 kW. The battery's target SOC trades downward against upward
    flexibility (90 %: 1 201 / 0 kW; 50 %: 1 192 / 250 kW).
- **Edge band observations (to check with the edge authors):**
  - The job band is ≈ 0 in S1.2 (0 kW down, 4 kW up on average).
  - The band reference at horizon step 1 is on average 464 kW below the observed value at step 0, so bands for 30 s
    ahead and beyond are anchored ≈ 10 % low.
- **Islanding is expressible, but the S1.2 site cannot island.** PV, the battery and full curtailment of the flexible
  load cover 39 % of the demand during a 1-h island. The built-in policy aborts at 11:30. A 1 000 kW per step ramp on
  reconnection starts from 0 kW and adds 45 min and 1.2 MWh of shedding.
- **Effort:** 146 lines of custom code (deferrable load 28, slack 9, policy 77, event controller 10, band 22). 210
  Vessim runs took 3.6 s in total.